In [7]:
!pip install streamlit pyngrok -q

In [8]:
import subprocess, threading, os, time, shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

PROJET_DIR = "/content/drive/MyDrive/MachineLearningProject"
MODELS_DIR = f"{PROJET_DIR}/models"

os.system("pkill -9 -f streamlit")
os.system("pkill -9 -f cloudflared")
os.system("pkill -9 -f lt")
time.sleep(2)

# Installer cloudflared
os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared")
os.system("chmod +x /usr/local/bin/cloudflared")

# Copier fichiers
os.makedirs("/content/app", exist_ok=True)
shutil.copy(f"{MODELS_DIR}/modele_final.pkl", "/content/app/modele_final.pkl")
shutil.copy(f"{MODELS_DIR}/features.pkl",     "/content/app/features.pkl")
shutil.copy(f"{PROJET_DIR}/src/app.py",       "/content/app/app.py")
print("✅ Fichiers copiés")

# Lancer Streamlit
def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py",
                    "--server.port=8501", "--server.headless=true"],
                   cwd="/content/app")
threading.Thread(target=run_streamlit, daemon=True).start()

print("⏳ Démarrage Streamlit...")
time.sleep(12)  # attendre que Streamlit soit prêt

# Lancer cloudflared et récupérer l'URL
import re
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

print("⏳ Création du tunnel...")
url = None
for _ in range(30):
    line = proc.stderr.readline()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print(f"\n🌐 App disponible ici : {url}")
print("(pas besoin de mot de passe avec cloudflared)")

Mounted at /content/drive
✅ Fichiers copiés
⏳ Démarrage Streamlit...
⏳ Création du tunnel...

🌐 App disponible ici : https://tobago-mounting-villas-childrens.trycloudflare.com
(pas besoin de mot de passe avec cloudflared)
